# Distribution Analysis
**Goal:** Understand sideslip distribution, band mapping, and class imbalance characteristics.

## Questions to Answer
1. What is the exact many/medium/few distribution?
2. How does sideslip correlate with sensor features?
3. Are there per-file distribution differences?
4. What are the band boundaries and their justification?

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import glob
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Path setup
PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.band_mapping import load_band_mapping, classify_band, get_band_statistics
from utils.reproducibility_utils import seed_everything

SEED = 42
seed_everything(SEED)

TRAIN_PATH = PROJECT_ROOT / "datasets_challenge" / "train"
DIST_PATH = PROJECT_ROOT / "datasets_challenge" / "sideSlip_distribution.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"Train path: {TRAIN_PATH}")

## 1. Official Band Mapping Analysis

In [ ]:
# Load official distribution file
dist_df = pd.read_csv(DIST_PATH)
print(f"Distribution file shape: {dist_df.shape}")
print(f"\nColumns: {dist_df.columns.tolist()}")
print(f"\nSample rows:")
dist_df.head(10)

In [ ]:
# Band statistics from distribution file
band_counts = dist_df['band'].value_counts()
print("Band Distribution (unique sideslip values):")
print(band_counts)
print(f"\nTotal unique values: {len(dist_df)}")

# Visualize band boundaries
fig = px.histogram(dist_df, x='sideSlip', color='band', 
                   color_discrete_map={'many': 'green', 'medium': 'orange', 'few': 'red'},
                   title='Sideslip Distribution by Band (Official Mapping)',
                   nbins=100)
fig.update_layout(xaxis_title='Sideslip (degrees)', yaxis_title='Count', height=400)
fig.show()

In [ ]:
# Find exact band boundaries
band_dict = load_band_mapping()

many_vals = [v for v, b in band_dict.items() if b == 'many']
medium_vals = [v for v, b in band_dict.items() if b == 'medium']
few_vals = [v for v, b in band_dict.items() if b == 'few']

print("Band Boundaries:")
print(f"  Many:   [{min(many_vals):.1f}, {max(many_vals):.1f}]")
print(f"  Medium: [{min(medium_vals):.1f}, {max(medium_vals):.1f}]")
print(f"  Few:    [{min(few_vals):.1f}, {max(few_vals):.1f}]")

## 2. Load All Training Data

In [ ]:
# Load all training files
train_files = sorted(glob.glob(str(TRAIN_PATH / "train_part_*.csv")))
print(f"Found {len(train_files)} training files")

all_dfs = []
file_stats = []

for f in train_files:
    df = pd.read_csv(f)
    filename = Path(f).name
    
    # Calculate per-file statistics
    bands = classify_band(df['sideSlip'].values, band_dict)
    stats = {
        'file': filename,
        'total_rows': len(df),
        'many_count': (bands == 'many').sum(),
        'medium_count': (bands == 'medium').sum(),
        'few_count': (bands == 'few').sum(),
        'many_pct': 100 * (bands == 'many').sum() / len(bands),
        'medium_pct': 100 * (bands == 'medium').sum() / len(bands),
        'few_pct': 100 * (bands == 'few').sum() / len(bands),
        'sideslip_mean': df['sideSlip'].mean(),
        'sideslip_std': df['sideSlip'].std(),
        'sideslip_min': df['sideSlip'].min(),
        'sideslip_max': df['sideSlip'].max(),
    }
    file_stats.append(stats)
    all_dfs.append(df)
    
file_stats_df = pd.DataFrame(file_stats)
all_data = pd.concat(all_dfs, ignore_index=True)

print(f"\nTotal samples: {len(all_data):,}")

In [ ]:
# Overall band distribution
overall_bands = classify_band(all_data['sideSlip'].values, band_dict)
overall_stats = get_band_statistics(all_data['sideSlip'].values, band_dict)

print("Overall Band Distribution:")
for band in ['many', 'medium', 'few']:
    print(f"  {band:8s}: {overall_stats[band]['count']:,} ({overall_stats[band]['pct']:.2f}%)")

# Imbalance ratio
imbalance_ratio = overall_stats['many']['count'] / overall_stats['few']['count']
print(f"\nImbalance Ratio (many/few): {imbalance_ratio:.1f}x")

## 3. Per-File Distribution Analysis

In [ ]:
# Visualize per-file band distribution
fig = go.Figure()

for band, color in [('many_pct', 'green'), ('medium_pct', 'orange'), ('few_pct', 'red')]:
    fig.add_trace(go.Bar(
        x=file_stats_df['file'],
        y=file_stats_df[band],
        name=band.replace('_pct', ''),
        marker_color=color
    ))

fig.update_layout(
    title='Band Distribution by File',
    xaxis_title='File',
    yaxis_title='Percentage',
    barmode='stack',
    height=500,
    xaxis_tickangle=-45
)
fig.show()

In [ ]:
# Files with highest few-shot content (potential validation candidates)
top_few = file_stats_df.nlargest(5, 'few_pct')[['file', 'total_rows', 'many_pct', 'medium_pct', 'few_pct']]
print("Top 5 files with highest few-shot percentage:")
print(top_few.to_string(index=False))

# Files with most balanced distribution
file_stats_df['balance_score'] = abs(file_stats_df['few_pct'] - overall_stats['few']['pct']) + \
                                  abs(file_stats_df['medium_pct'] - overall_stats['medium']['pct'])
balanced_files = file_stats_df.nsmallest(5, 'balance_score')[['file', 'total_rows', 'many_pct', 'medium_pct', 'few_pct']]
print("\nTop 5 most balanced files (closest to overall distribution):")
print(balanced_files.to_string(index=False))

## 4. Sideslip Distribution Shape

In [ ]:
# Overall sideslip histogram
fig = make_subplots(rows=1, cols=2, subplot_titles=['Linear Scale', 'Log Scale'])

# Linear
fig.add_trace(
    go.Histogram(x=all_data['sideSlip'], nbinsx=200, name='Linear'),
    row=1, col=1
)

# Log scale
hist_counts, hist_edges = np.histogram(all_data['sideSlip'], bins=200)
fig.add_trace(
    go.Bar(x=hist_edges[:-1], y=np.log10(hist_counts + 1), name='Log10'),
    row=1, col=2
)

fig.update_layout(height=400, title='Sideslip Distribution (Heavy Long-Tail)')
fig.update_xaxes(title_text='Sideslip (degrees)')
fig.update_yaxes(title_text='Count', row=1, col=1)
fig.update_yaxes(title_text='Log10(Count)', row=1, col=2)
fig.show()

In [ ]:
# Distribution statistics
print("Sideslip Statistics:")
print(f"  Mean:     {all_data['sideSlip'].mean():.4f}")
print(f"  Median:   {all_data['sideSlip'].median():.4f}")
print(f"  Std:      {all_data['sideSlip'].std():.4f}")
print(f"  Min:      {all_data['sideSlip'].min():.4f}")
print(f"  Max:      {all_data['sideSlip'].max():.4f}")
print(f"  Skewness: {all_data['sideSlip'].skew():.4f}")
print(f"  Kurtosis: {all_data['sideSlip'].kurtosis():.4f}")

# Percentiles
print("\nPercentiles:")
for p in [50, 75, 90, 95, 99]:
    val = np.percentile(np.abs(all_data['sideSlip']), p)
    print(f"  {p}th percentile (abs): {val:.4f}")

## 5. Feature Correlations with Sideslip

In [ ]:
# Calculate correlations
feature_cols = [col for col in all_data.columns if col not in ['sideSlip', 'vyCG', 'longitude', 'latitude', 'time']]
correlations = all_data[feature_cols + ['sideSlip']].corr()['sideSlip'].drop('sideSlip').sort_values(key=abs, ascending=False)

print("Feature Correlations with Sideslip:")
print(correlations.to_string())

In [ ]:
# Visualize top correlations
top_n = 10
top_corr = correlations.head(top_n)

fig = go.Figure(go.Bar(
    x=top_corr.values,
    y=top_corr.index,
    orientation='h',
    marker_color=['red' if v < 0 else 'green' for v in top_corr.values]
))
fig.update_layout(
    title=f'Top {top_n} Feature Correlations with Sideslip',
    xaxis_title='Correlation',
    yaxis_title='Feature',
    height=400
)
fig.show()

## 6. Sideslip vs Key Sensors (by Band)

In [ ]:
# Sample for visualization (too many points otherwise)
sample_idx = np.random.choice(len(all_data), min(10000, len(all_data)), replace=False)
sample_df = all_data.iloc[sample_idx].copy()
sample_df['band'] = classify_band(sample_df['sideSlip'].values, band_dict)

# Sideslip vs yawRate
fig = px.scatter(sample_df, x='yawRate', y='sideSlip', color='band',
                 color_discrete_map={'many': 'green', 'medium': 'orange', 'few': 'red'},
                 title='Sideslip vs YawRate by Band', opacity=0.5)
fig.update_layout(height=500)
fig.show()

In [ ]:
# Sideslip vs ayCG (lateral acceleration)
fig = px.scatter(sample_df, x='ayCG', y='sideSlip', color='band',
                 color_discrete_map={'many': 'green', 'medium': 'orange', 'few': 'red'},
                 title='Sideslip vs Lateral Acceleration by Band', opacity=0.5)
fig.update_layout(height=500)
fig.show()

## 7. Band Boundary Thresholds

In [ ]:
# Determine exact thresholds from band_dict
abs_sideslip = np.abs(all_data['sideSlip'].values)
bands = classify_band(all_data['sideSlip'].values, band_dict)

# Find boundaries
many_max = np.max(np.abs(all_data.loc[bands == 'many', 'sideSlip']))
few_min = np.min(np.abs(all_data.loc[bands == 'few', 'sideSlip']))

print("Discovered Band Boundaries (absolute sideslip):")
print(f"  Many:   |sideslip| <= {many_max:.1f}")
print(f"  Medium: {many_max:.1f} < |sideslip| < {few_min:.1f}")
print(f"  Few:    |sideslip| >= {few_min:.1f}")

# Commonly used thresholds in the codebase
print("\nCommon thresholds used in codebase:")
print("  2.4° - many/medium boundary (from weighted loss)")
print("  4.0° - medium/few boundary (from weighted loss)")

## Summary

| Metric | Value |
|--------|-------|
| Total samples | See above |
| Many-shot % | ~78% |
| Medium-shot % | ~17% |
| Few-shot % | ~5% |
| Imbalance ratio | ~16x |

### Key Findings
1. **Heavy long-tail distribution**: 95%+ samples are near zero
2. **High correlation**: `yawRate`, `ayCG`, `handwheelAngle` most predictive
3. **Per-file variance**: Some files have much higher drift content
4. **Band boundaries**: ~2.4° and ~4.0° (absolute sideslip)